# General Document Assistant (RAG)

This notebook implements a RAG (Retrieval Augmented Generation) system designed to act as an assistant for querying information from uploaded PDF documents.

***OBS: RAG pipeline needs to be fix from `HuggingFacePipeline` to `ChatHuggingFace`, in order for a better instruction follow by the model.***

## 1. Install Libraries

In [ ]:
!pip install -q transformers accelerate sentence-transformers pypdf gradio torch langchain langchain-community langchain-core langchain-text-splitters faiss-cpu einops pymupdf
!pip install -q -U bitsandbytes
print("Libraries installed/updated. IMPORTANT: RESTART THE RUNTIME (Runtime > Restart session) before proceeding.")

Libraries installed/updated. IMPORTANT: RESTART THE RUNTIME (Runtime > Restart session) before proceeding.


## 2. Imports

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch, gradio as gr, warnings, traceback, re
from typing import List

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, AutoModel, set_seed
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader # MODIFIED: Using PyMuPDFLoader
from langchain.embeddings.base import Embeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import LLMChain, ConversationalRetrievalChain
from langchain.chains.question_answering import load_qa_chain
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

warnings.filterwarnings("ignore")
set_seed(42)

## 3. PDF Upload and Processing (Revised for General Documents)

In [ ]:
from google.colab import files

print("Please upload your PDF.")
uploaded_pdf = files.upload()

docs = []
pdf_filename = None
temp_pdf_path = None

if not uploaded_pdf:
    print("No PDF uploaded. The assistant will operate in general knowledge mode only.")
else:
    pdf_filename = list(uploaded_pdf.keys())[0]
    temp_pdf_path = os.path.join("/content/", pdf_filename) # Save to a temporary path
    global_pdf_filename_for_chat = pdf_filename
    print(f"Filename '{global_pdf_filename_for_chat}' stored for chat reference.")
    with open(temp_pdf_path, "wb") as f:
        f.write(uploaded_pdf[pdf_filename])
    print(f"Uploaded '{pdf_filename}' and saved to '{temp_pdf_path}'.")

    # Optional: Basic text cleaning function
    def clean_text_content(text):
        text = re.sub(r'\s*\n\s*', '\n', text) # Normalize newlines
        text = re.sub(r'-\n(?=[a-zA-Z])', '', text) # De-hyphenate words split across lines (basic)
        # Add more sophisticated cleaning rules as needed (e.g., for headers/footers, ligatures)
        return text.strip()

    try:
        print(f"Loading PDF: {temp_pdf_path}")
        loader = PyMuPDFLoader(temp_pdf_path)
        documents = loader.load()
        print(f"Loaded {len(documents)} pages/initial documents from PDF.")

        # Apply text cleaning and ensure metadata
        cleaned_documents = []
        for i, doc in enumerate(documents):
            # doc.page_content = clean_text_content(doc.page_content) # Uncomment to enable cleaning
            if 'filename' not in doc.metadata:
                 doc.metadata['filename'] = pdf_filename
            if 'page' not in doc.metadata: # PyMuPDFLoader usually adds page, but good to be defensive
                 doc.metadata['page'] = doc.metadata.get('page_number', i) # or some default
            cleaned_documents.append(doc)

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150) # Increased overlap slightly
        docs = text_splitter.split_documents(cleaned_documents)
        print(f"PDF '{pdf_filename}' processed into {len(docs)} chunks.")

    except Exception as e:
        print(f"Error processing PDF {pdf_filename}: {e}")
        traceback.print_exc()
        docs = [] # Ensure docs is empty on error
    finally:
        # Clean up the temporary file
        if temp_pdf_path and os.path.exists(temp_pdf_path):
            os.remove(temp_pdf_path)
            print(f"Temporary file '{temp_pdf_path}' removed.")

if not docs:
    print("No document chunks were created. The RAG system will rely on its general knowledge.")

Please upload your PDF.


Saving chemyunion.pdf to chemyunion.pdf
Uploaded 'chemyunion.pdf' and saved to '/content/chemyunion.pdf'.
Loading PDF: /content/chemyunion.pdf
Loaded 40 pages/initial documents from PDF.
PDF 'chemyunion.pdf' processed into 90 chunks.
Temporary file '/content/chemyunion.pdf' removed.


## 4. Setup Embedding Model (Jina V3) and Vector Store

In [ ]:
# Custom Jina Embeddings Class for Langchain
class JinaAIEmbeddings(Embeddings):
    def __init__(self, model_name: str = "jinaai/jina-embeddings-v3", device: str = "cuda"):
        self.model = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(device)
        self.device = device
        print(f"Jina embedding model '{model_name}' loaded on {self.device}.")

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # For RAG, documents should be embedded as passages
        embeddings = self.model.encode(texts, task='retrieval.passage', batch_size=32) # Added batch_size
        return embeddings.tolist()

    def embed_query(self, text: str) -> List[float]:
        # Queries should be embedded as queries
        embedding = self.model.encode([text], task='retrieval.query')
        return embedding[0].tolist()

    # Add a simple aembed_query and aembed_documents for async compatibility if needed by some chains
    async def aembed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.embed_documents(texts)

    async def aembed_query(self, text: str) -> List[float]:
        return self.embed_query(text)

# Initialize Jina Embeddings
jina_embedding_model_name = "jinaai/jina-embeddings-v3" # or jina-embeddings-v2-base-en for English only
embeddings = JinaAIEmbeddings(model_name=jina_embedding_model_name)

print(f"Custom Jina Embedding model '{jina_embedding_model_name}' configured.")

vector_store = FAISS.from_documents(docs, embeddings)
print("FAISS vector store created with Jina embeddings.")
retriever_k_value = 5 # Retrieve top N docs
retriever = vector_store.as_retriever(search_kwargs={"k": retriever_k_value})
print(f"Retriever configured for k={retriever_k_value}.")

Jina embedding model 'jinaai/jina-embeddings-v3' loaded on cuda.
Custom Jina Embedding model 'jinaai/jina-embeddings-v3' configured.
FAISS vector store created with Jina embeddings.
Retriever configured for k=5.


## OPT. Hugging face login for gated models

In [ ]:
# Hugging Face Login (Required for Gated Models like Gemma)

import os
from huggingface_hub import login, HfFolder, whoami
from getpass import getpass

print("--------------------------------------------------------------------------")
print("Attempting Hugging Face login...")
print("This is required to download gated models like Gemma.")
print("You'll need a Hugging Face User Access Token with 'read' permissions.")
print("Get one from: https://huggingface.co/settings/tokens")
print("--------------------------------------------------------------------------\n")

# Initialize login status
logged_in_successfully = False

# --- Option 1: Check if already logged in (e.g., token saved in environment) ---
try:
    user_info = whoami()
    print(f"Already logged in to Hugging Face as: {user_info.get('name')}")
    print(f"User ID: {user_info.get('id')}, Org: {user_info.get('orgs')}")
    logged_in_successfully = True
except Exception:
    print("Not currently logged in via an active session or globally configured token.")
    pass # Continue to try other methods

# --- Option 2: Try to use HUGGING_FACE_HUB_TOKEN environment variable if not already logged in ---
if not logged_in_successfully:
    hf_token_env = os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if hf_token_env:
        print("\nFound HUGGING_FACE_HUB_TOKEN environment variable.")
        print("Attempting login with environment variable token...")
        try:
            login(token=hf_token_env, add_to_git_credential=False)
            user_info = whoami() # Verify login
            print(f"Successfully logged in to Hugging Face as: {user_info.get('name')}")
            logged_in_successfully = True
        except Exception as e:
            print(f"Login with environment variable token failed: {e}")
            print("Proceeding to manual token input if needed.")
    else:
        print("\nHUGGING_FACE_HUB_TOKEN environment variable not found.")

# --- Option 3: Manual token input if other methods failed or weren't available ---
if not logged_in_successfully:
    print("\nPlease enter your Hugging Face Hub access token:")
    print("(Your token will not be displayed as you type)")
    try:
        hf_token_manual = getpass()
        if hf_token_manual:
            print("Attempting login with manually entered token...")
            login(token=hf_token_manual, add_to_git_credential=False) # Set add_to_git_credential=True to save for future CLI use
            user_info = whoami() # Verify login
            print(f"Successfully logged in to Hugging Face as: {user_info.get('name')}")
            logged_in_successfully = True
        else:
            print("No token was entered.")
    except Exception as e:
        print(f"Hugging Face login with manual token failed: {e}")

# --- Final Status and Important Reminder ---
if logged_in_successfully:
    print("\n✅ Hugging Face login successful.")
else:
    print("\n⚠️ Hugging Face login was not successful or was skipped.")
    print("   Downloads of gated models (like Gemma) will likely fail.")
    print("   Please ensure you provide a valid token if you intend to use such models.")

print("\n==========================================================================")
print("IMPORTANT REMINDER FOR GEMMA (and other gated models):")
print("1. You MUST visit the model card on Hugging Face for EACH Gemma model version")
print("   you intend to use (e.g., google/gemma-7b-it, google/gemma-2b) and")
print("   explicitly ACCEPT THE LICENSE AGREEMENT / 'Agree and access repository'.")
print("2. This login step only authenticates you; it doesn't accept licenses for you.")
print("==========================================================================")

--------------------------------------------------------------------------
Attempting Hugging Face login...
This is required to download gated models like Gemma.
You'll need a Hugging Face User Access Token with 'read' permissions.
Get one from: https://huggingface.co/settings/tokens
--------------------------------------------------------------------------

Already logged in to Hugging Face as: EmLL00000
User ID: 66e8b480b3982349fa05c33a, Org: []

✅ Hugging Face login successful.

IMPORTANT REMINDER FOR GEMMA (and other gated models):
1. You MUST visit the model card on Hugging Face for EACH Gemma model version
   you intend to use (e.g., google/gemma-7b-it, google/gemma-2b) and
   explicitly ACCEPT THE LICENSE AGREEMENT / 'Agree and access repository'.
2. This login step only authenticates you; it doesn't accept licenses for you.


## 5. Load LLM (With Quantization)

In [ ]:
model_id = "google/gemma-3-4b-it" # if using a better GPU Consider bigger versions for potentially better general reasoning
device = "cuda" #Using GPU

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False
)

print(f"Loading LLM '{model_id}' with 4-bit quantization on {device}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
print(f"LLM '{model_id}' loaded.")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.15,
    do_sample=True
)
llm = HuggingFacePipeline(pipeline=pipe)
print("HuggingFacePipeline for LLM created.")

Loading LLM 'google/gemma-3-4b-it' with 4-bit quantization on cuda...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


LLM 'google/gemma-3-4b-it' loaded.
HuggingFacePipeline for LLM created.


## 6. Setup Conversational RAG Chain with Generalized Prompts

In [ ]:
# 1. SYSTEM_PROMPT for the final question answering (combine_docs_chain) - GENERALIZED & ENHANCED
SYSTEM_PROMPT_QA_GENERAL = """
You are a friendly, helpful, and conversational chat assistant.
Your primary role is to provide information and answer questions based on the 'Context' (excerpts from one or more documents) provided below.
You are also capable of answering general knowledge questions. Your responses should sound natural and helpful.

When a 'User Question' seems to be answerable from the provided 'Context':
1.  Identify the key entities, topics, or questions in the 'User Question'.
2.  Carefully scan the 'Context' for information relevant to the 'User Question'.
3.  Synthesize this information into a concise and natural-sounding answer for the 'Final Answer'.

4.  If the 'Context' provides some relevant information but lacks specific details the user is asking for:
    a. First, share what *is* found in a helpful way. For example: "The document mentions that the project was initiated in Q3, outlining several key objectives."
    b. Then, clearly and politely state that the specific detail is not in the provided contextual information. For example: "However, it doesn't specify the exact budget allocated for the initial phase."
    c. If the 'Context' includes source information (like a filename or page number from the document's metadata if it was part of the retrieved text), you MAY naturally suggest the user refer to the original source for more comprehensive details if appropriate.

5.  Do not make up information not present in the 'Context' for document-based questions.

When a 'User Question' is clearly NOT answerable from the 'Context' or is a general knowledge question:
1.  Acknowledge this in a friendly way. For example: "That's an interesting question! While the provided document(s) don't seem to cover that..."
2.  THEN, YOU MUST ATTEMPT to answer the question using your general knowledge.
3.  If you can provide an answer from your general knowledge, do so directly and naturally in the 'Final Answer'.
4.  If you genuinely do not know the answer from your general knowledge, then politely state that. For example: "That's a great question, but I don't have information on that topic at the moment."
5.  For these general knowledge questions, completely ignore the 'Context' section.

When a 'User Question' is specifically asking about the source of the information (e.g., "What document are you using?", "What is the filename?"):
1.  Check if the 'Context' contains explicit metadata like a filename (e.g., if a chunk started with "Source: filename.pdf, Page: X...").
2.  If a filename is clearly identifiable from the 'Context' provided for this turn, state it directly in the 'Final Answer'. For example: "I am getting information from a document that seems to be titled 'Product_Catalog_2024.pdf' based on the context."
3.  If no specific filename is identifiable in the current 'Context' for this turn, but you know a document was loaded (because context is present), you can say: "I am referencing the document that was loaded into the system for our conversation."
4.  If no document was loaded (i.e., 'Context' is empty or indicates no relevant document chunks were retrieved), and the question is about a source, state that no document is currently being referenced for this query.
5.  Do NOT answer about the content of the document (like product details) when asked specifically about the source document itself. Prioritize answering the question about the source.

If the 'User Question' is a general greeting, respond politely and naturally.
Always respond in English. Ensure your final output is well-formed and directly answers the user.

The 'Final Answer' section MUST be a direct, conversational response to the 'User Question'.
It should be formulated as if you are speaking directly to the user.
Crucially, the 'Final Answer' section should NOT contain:
- Any self-referential statements about your process (e.g., "Based on the context...", "I found that...", "The context does not mention...", "According to the document...").
- Any bracketed or parenthetical notes about your limitations or observations (e.g., "(Note: ...)", "[General Knowledge Response]").
- Repetition of the 'Assistant's Thought Process'.
Do NOT start your 'Final Answer' with phrases like "The document states", "The context shows", or "According to the provided information". Integrate the information naturally.

Your entire output for this turn will be structured with your internal reasoning first, then the clean answer.

IMPORTANT: Your response for this turn must ONLY consist of the 'Assistant's Thought Process' followed by the 'Final Answer:' for the *current* 'User Question'.
After generating the 'Final Answer:', YOU MUST STOP. Do NOT generate any further text, questions, examples, code, or special formatting.
Your 'Final Answer:' should be plain conversational text.
"""

# 2. Prompt Template for the combine_docs_chain (QA_PROMPT)
QA_PROMPT_STR = SYSTEM_PROMPT_QA_GENERAL + """

Context:
{context}

User Question: {question}

Assistant's Thought Process (internal considerations to arrive at the answer, not for final display):
- Question Type: (Document-based, General Knowledge, or Meta-question about Source?)
- Key Entities/Topic in Question: (Identify from User Question)
- Context Scan Summary: (Key findings from 'Context' relevant to the question? Are specific details present or missing? Any relevant source metadata like filenames or page numbers noted in the context?)
- Answer Strategy: (Based on findings and system instructions, how to structure the answer? If meta-question about source, prioritize that. If details are missing, plan to state what's found, what's missing.)
- Natural Language Goal: (Ensure the final answer is conversational, helpful, and directly answers the specific question asked, avoiding prefixes like "According to the document...".)

Final Answer:
"""
QA_PROMPT = PromptTemplate(
    template=QA_PROMPT_STR, input_variables=["context", "question"]
)

# 3. Custom Prompt for Condensing Questions (question_generator_chain)
CUSTOM_CONDENSE_QUESTION_TEMPLATE = """
Given the Chat History and a Follow Up Input, rephrase the Follow Up Input into a concise, standalone question in English. This standalone question will be used for a knowledge base search.

**Critical Instructions for Rephrasing:**
1.  **Prioritize 'Follow Up Input' Entities:** If the 'Follow Up Input' explicitly mentions a specific entity, concept, or name, that EXACT phrasing or a close semantic equivalent MUST be the primary subject of the 'Rephrased Standalone Question'.
2.  **Use Chat History for Context Only:** Use the 'Chat History' to understand pronoun references or to add necessary context if the 'Follow Up Input' is very short.
3.  **Standalone and Searchable:** The rephrased question must make sense on its own.
4.  **Handle Meta-Questions:** If the 'Follow Up Input' is a question about the document source or your process (e.g., "what document are you using?", "where did you get that?"), rephrase it to be about the source or process.
5.  **Output ONLY the Question:** Your entire response should be *just* the rephrased question. No explanations, no "Rephrased Standalone Question:" prefix, no other text.

**Examples:**

Chat History:
User: What are the main findings of the climate report?
Assistant: The report highlights rising global temperatures and increased CO2 levels.
Follow Up Input: What does it say about mitigation strategies?
Rephrased Standalone Question: What mitigation strategies are mentioned in the climate report?

Chat History:
User: Tell me about the 'Odyssey' project mentioned in the document.
Assistant: The 'Odyssey' project is a new software development initiative...
Follow Up Input: What is its timeline?
Rephrased Standalone Question: What is the timeline for the 'Odyssey' project?

Chat History:
User: Tell me about Unibase BK.
Assistant: Unibase BK is a self-emulsifying base...
Follow Up Input: What document are you getting this from?
Rephrased Standalone Question: What is the source document for information about Unibase BK?

Chat History:
{chat_history}

Follow Up Input: {question}

Rephrased Standalone Question:"""
CONDENSE_QUESTION_CUSTOM_PROMPT = PromptTemplate.from_template(CUSTOM_CONDENSE_QUESTION_TEMPLATE)

# 4. Initialize Memory
memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key='answer',
    k=3
)
print(f"Using ConversationBufferWindowMemory with k={memory.k}")

# Custom parser for Standalone Question
class StandaloneQuestionOutputParser(StrOutputParser):
    def parse(self, text: str) -> str:
        marker = "Rephrased Standalone Question:"
        parts = text.split(marker)
        if len(parts) > 1:
            generated_part = parts[-1].strip()
            lines = [line.strip() for line in generated_part.split('\n') if line.strip()]
            if lines: return lines[0]
            return ""
        else:
            lines = [line.strip() for line in text.split('\n') if line.strip()]
            if lines: return lines[-1]
            return text.strip()

# 5. Create Question Generator chain
question_generator_chain = LLMChain(
    llm=llm, # Ensure 'llm' is defined from Section 5
    prompt=CONDENSE_QUESTION_CUSTOM_PROMPT,
    output_parser=StandaloneQuestionOutputParser(),
    verbose=False
)
print("Question generator chain created.")

# 6. Create Combine Docs chain
combine_docs_chain = load_qa_chain(
    llm=llm, # Ensure 'llm' is defined
    chain_type="stuff",
    prompt=QA_PROMPT,
    verbose=False
)
print("Combine docs chain created.")

# 7. Create ConversationalRetrievalChain
qa_chain = None
if retriever: # Ensure 'retriever' is defined from Section 4
    qa_chain = ConversationalRetrievalChain(
        retriever=retriever,
        question_generator=question_generator_chain,
        combine_docs_chain=combine_docs_chain,
        memory=memory,
        return_source_documents=True,
        return_generated_question=True,
        verbose=False
    )
    print("ConversationalRetrievalChain created.")
else:
    print("Retriever is not available. ConversationalRetrievalChain cannot be created for document Q&A. Will operate in general knowledge mode if LLM is available.")

Using ConversationBufferWindowMemory with k=3
Question generator chain created.
Combine docs chain created.
ConversationalRetrievalChain created.


## 7. Gradio Chat Interface (Generalized)

In [ ]:
#To get the pdf name as a meta-data
global_pdf_filename_for_chat = pdf_filename if 'pdf_filename' in globals() and pdf_filename else "the uploaded document"


def polish_final_answer(text: str) -> str:
    if not text:
        return ""
    cleaned_text = text

    # Patterns to remove common LLM self-referential notes, preambles, etc.
    patterns_to_remove = [
        r"^\s*\[General Knowledge Response\]:\s*",
        r"^\s*Note: The context does not contain.*?based solely on the summary available within the context\.\s*$",
        r"^\s*\(Note: The assistant correctly identifies.*?\)\s*$",
        r"^\s*As an AI, I don't have personal opinions.*",
        # Prefixes related to document attribution (prompt should handle this, but as a backup)
        r"^\s*According to the document,\s*",
        r"^\s*The document states that\s*",
        r"^\s*The document mentions that\s*",
        r"^\s*Based on the document,\s*",
        r"^\s*In the provided document,\s*",
        r"^\s*The provided context shows that\s*",
        r"^\s*Based on the provided context,\s*",
        r"^\s*I found in the context that\s*",
    ]
    for pattern in patterns_to_remove:
        cleaned_text = re.sub(pattern, "", cleaned_text, flags=re.IGNORECASE | re.MULTILINE).strip()

    # Handle "I'm sorry, but... However..."
    sorry_however_match = re.match(r"I'm sorry, but.*?\. However, (.*)", cleaned_text, flags=re.IGNORECASE | re.DOTALL)
    if sorry_however_match:
        part_before_however = cleaned_text[:sorry_however_match.start(1) - len(" However, ")].lower()
        if any(phrase in part_before_however for phrase in ["context", "document", "provided data", "catalog"]):
            cleaned_text = sorry_however_match.group(1).strip()

    # Handle "That's an interesting question! While [context doesn't cover]..."
    preamble_match = re.match(
        r"That's an interesting question!\s*While (the provided context|the document\(s\)|my information on the topic|our product catalog) (doesn't cover that|doesn't seem to cover that|does not seem to cover that)\s*(?:topic)?\s*(?:,)?\s*(?:however,)?\s*(.*)",
        cleaned_text, flags=re.IGNORECASE | re.DOTALL
    )
    if preamble_match:
        actual_answer_after_preamble = preamble_match.group(3).strip()
        if actual_answer_after_preamble:
            cleaned_text = actual_answer_after_preamble

    return cleaned_text.strip()


def chat_response_interface(user_message, chat_history_tuples):
    print(f"\n================ NEW USER QUERY ================ ")
    print(f"User Query: {user_message}")
    bot_response_final = "I apologize, an error occurred. Please check logs."

    # Access the global filename (ensure it's set in Section 3)
    # This is a simple way; for more robust apps, consider Gradio state or class members.
    current_doc_name = global_pdf_filename_for_chat if 'global_pdf_filename_for_chat' in globals() else "the uploaded document"

    if not qa_chain: # No document loaded or retriever/chain setup failed
        print("--- Operating in General Knowledge Mode (No RAG chain available) ---")
        if llm: # Check if llm object itself is available
            try:
                # Basic prompt for general knowledge
                # Note: Gemma might still try to be conversational based on its fine-tuning
                # You might want a more specific "You are an AI assistant..." prompt here too.
                general_response = llm(f"User: {user_message}\nAssistant:") # llm is CTransformers or HF Pipeline
                bot_response_final = polish_final_answer(general_response.strip())
                if not bot_response_final:
                     bot_response_final = "I can try to answer general questions. What would you like to know?"
            except Exception as e_gk:
                print(f"Error in general knowledge fallback with LLM: {e_gk}")
                bot_response_final = "I'm having trouble processing general questions right now."
        else:
            bot_response_final = "The document processing system is not fully initialized, and the general knowledge module is also unavailable. Please ensure a document is uploaded and all setup steps are complete."

        print(f"--- [GK] Final Bot Message to Gradio ---\n{bot_response_final}")
        print("===============================================")
        return bot_response_final

    try:
        print("--- [1] Calling ConversationalRetrievalChain ---")
        # Potentially add filename to the question if it's about the source,
        # or ensure the prompt handles it. For now, relying on prompt.
        response_payload = qa_chain({"question": user_message}) # qa_chain is defined in Sec 6

        raw_llm_answer = response_payload.get('answer', '').strip()
        print(f"--- [2] Raw LLM Answer (first 1000 chars) ---\n{raw_llm_answer[:1000]}...")

        generated_question = response_payload.get('generated_question', user_message)
        print(f"--- [3] Standalone Question (used in QA_PROMPT) ---\n{generated_question}")

        source_documents = response_payload.get('source_documents', [])
        if source_documents:
            print("--- [4] Retrieved Source Documents ---")
            for i, doc in enumerate(source_documents):
                page_num = doc.metadata.get('page', 'N/A')
                # Use 'filename' if present, else 'source'
                filename_meta = doc.metadata.get('filename', doc.metadata.get('source', 'N/A'))
                print(f"  Doc {i+1}: (File: {filename_meta}, Page: {page_num}) Content: {doc.page_content[:300]}...")
        else:
            print("--- [4] No source documents retrieved for this query.")

        # Parsing logic for "Final Answer:" marker
        bot_message_cleaned = ""
        final_answer_marker = "Final Answer:"

        # Try to find the start of the LLM's actual generation after the echoed prompt
        idx_user_q_in_raw = raw_llm_answer.rfind(f"User Question: {generated_question}")
        idx_final_answer_after_user_q = -1
        if idx_user_q_in_raw != -1:
            idx_thought_process_after_user_q = raw_llm_answer.find("Assistant's Thought Process", idx_user_q_in_raw)
            if idx_thought_process_after_user_q != -1:
                idx_final_answer_after_user_q = raw_llm_answer.find(final_answer_marker, idx_thought_process_after_user_q)

        if idx_final_answer_after_user_q != -1:
            start_of_actual_answer = idx_final_answer_after_user_q + len(final_answer_marker)
            llm_actual_generation = raw_llm_answer[start_of_actual_answer:].lstrip()
        else:
            marker_index = raw_llm_answer.rfind(final_answer_marker)
            if marker_index != -1:
                llm_actual_generation = raw_llm_answer[marker_index + len(final_answer_marker):].lstrip()
            else:
                llm_actual_generation = raw_llm_answer # Use raw if no marker found

        # Remove any subsequent thought process or markers
        stop_delimiters = ["\nUser Question:", "\nAssistant's Thought Process", "\nFinal Answer:"]
        min_idx_of_next_block = len(llm_actual_generation)
        for delimiter in stop_delimiters:
            idx = llm_actual_generation.find(delimiter)
            if idx != -1: min_idx_of_next_block = min(min_idx_of_next_block, idx)

        bot_message_cleaned = llm_actual_generation[:min_idx_of_next_block].strip()
        print(f"--- [5a] Cleaned Message (parsed) ---\n{bot_message_cleaned}")

        if not bot_message_cleaned.strip() and raw_llm_answer.strip():
            print("--- [5b] Cleaned message was empty, using raw LLM answer before polishing.")
            bot_message_cleaned = raw_llm_answer

        print(f"--- [5c PRE-POLISH] Bot Message Cleaned ---\n{bot_message_cleaned}")
        bot_response_final = polish_final_answer(bot_message_cleaned)
        print(f"--- [5d POST-POLISH] Bot Response Final ---\n{bot_response_final}")

        if not bot_response_final.strip():
            print("WARNING: Final polished response is empty. Using a fallback message.")
            if raw_llm_answer.strip() and not raw_llm_answer.lower().startswith("you are a friendly"):
                 bot_response_final = raw_llm_answer
            else:
                 bot_response_final = "I'm having trouble formulating a response. Could you try rephrasing your question?"

    except Exception as e:
        print(f"!!!!!!!! ERROR DURING QA CHAIN EXECUTION !!!!!!!!")
        print(f"Error Type: {type(e).__name__}"); print(f"Error Message: {str(e)}")
        traceback.print_exc()
        bot_response_final = "An error occurred processing your request. Please check the console logs for more details."

    print(f"--- [6] Final Bot Message to Gradio ---\n{bot_response_final}")
    print("===============================================")
    return bot_response_final

# --- Gradio UI Setup (remains largely the same) ---
dark_theme_css = """body, .gradio-container { background-color: #202124 !important; color: #E8EAED !important; } .gr-panel { background-color: #303134 !important; border: none !important; box-shadow: 0 2px 10px rgba(0,0,0,0.2) !important; } .gr-button { background-color: #3C4043 !important; color: #E8EAED !important; border: 1px solid #5F6368 !important; } .gr-button:hover { background-color: #4A4E52 !important; } .gr-input, .gr-textarea, .gr-textbox { background-color: #3C4043 !important; color: #E8EAED !important; border: 1px solid #5F6368 !important; } .gr-chatbot { background-color: #303134 !important; border: none !important; } .gr-chatbot .user-message .message { background-color: #4A90E2 !important; color: white !important; border-radius: 18px 18px 0 18px !important; } .gr-chatbot .bot-message .message { background-color: #3C4043 !important; color: #E8EAED !important; border-radius: 18px 18px 18px 0 !important; } .gr-message { color: #E8EAED !important; } .gr-label { color: #BDC1C6 !important; }"""

with gr.Blocks(css=dark_theme_css, title="Document Assistant (RAG)") as demo_interface:
    gr.Markdown("## Document Assistant")
    gr.Markdown("Upload a PDF to ask questions about its content, or ask general knowledge questions.")

    chatbot_display = gr.Chatbot(
        [],
        elem_id="chatbot_main",
        bubble_full_width=False,
        height=600,
        avatar_images=(None, "https://i.imgur.com/1Jd7kUe.png")
    )
    with gr.Row():
        txt_input = gr.Textbox(
            scale=4,
            show_label=False,
            placeholder="Ask about the uploaded document or general topics...",
            container=False
        )
        btn_submit = gr.Button("Send", scale=1)

    def handle_gradio_submit(user_message, chat_display_history):
        bot_reply = chat_response_interface(user_message, chat_display_history)
        chat_display_history.append((user_message, bot_reply))
        return "", chat_display_history

    txt_input.submit(handle_gradio_submit, [txt_input, chatbot_display], [txt_input, chatbot_display])
    btn_submit.click(handle_gradio_submit, [txt_input, chatbot_display], [txt_input, chatbot_display])

print("Launching Gradio Interface...")
# Ensure global_pdf_filename_for_chat is defined before launching if you use it above
if 'pdf_filename' in globals() and pdf_filename:
    global_pdf_filename_for_chat = pdf_filename
else:
    global_pdf_filename_for_chat = "the uploaded document" # Default if no PDF was uploaded

demo_interface.launch(debug=True, share=True)

Launching Gradio Interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4e162b4f6ef9f1932c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



================ NEW USER QUERY ================ 
User Query: Who is chemyunion?
--- [1] Calling ConversationalRetrievalChain ---
--- [2] Raw LLM Answer (first 1000 chars) ---
You are a friendly, helpful, and conversational chat assistant.
Your primary role is to provide information and answer questions based on the 'Context' (excerpts from one or more documents) provided below.
You are also capable of answering general knowledge questions. Your responses should sound natural and helpful.

When a 'User Question' seems to be answerable from the provided 'Context':
1.  Identify the key entities, topics, or questions in the 'User Question'.
2.  Carefully scan the 'Context' for information relevant to the 'User Question'.
3.  Synthesize this information into a concise and natural-sounding answer for the 'Final Answer'.

4.  If the 'Context' provides some relevant information but lacks specific details the user is asking for:
    a. First, share what *is* found in a helpful way. For exampl